# Section 3: Extending RAG to an Agentic AI Workflow (25 Mins)

**Enterprise Use Case:** CloudCorp Smart Assistant (AskHR Agent)

**Notebook Goal:** Extend static RAG into an autonomous agent using Vertex AI Tool Calling with `FunctionDeclaration`. The agent reasons over user queries, retrieves policy context, and dynamically creates incident tickets (`INC-501`) or looks up live status.

In [1]:
# Step 3.0: Install Required Google Cloud Client Libraries
!pip install -q google-cloud-aiplatform>=1.60.0 google-cloud-storage>=2.14.0 numpy


zsh:1: 1.60.0 not found


In [7]:
# Step 3.1: Authenticate, Initialize Vertex AI & Load Policy Chunks from GCS
import os
import numpy as np

# 1. Load environment constants dynamically from .env file if available
def load_env_file():
    for path in [".env", "../.env", "cloudcorp-llms-on-gcp-lab/.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        val = v.strip().strip('"').strip("'")
                        if val:
                            os.environ.setdefault(k.strip(), val)
            print(f"Loaded configuration from '{path}'")
            return

load_env_file()

# 2. Project Configuration & Quota Project Settings
PROJECT_ID = os.environ.get("PROJECT_ID", "").strip() or os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip() or "your-gcp-project-id"  # @param {type:"string"}
REGION = os.environ.get("REGION", "us-central1").strip()               # @param {type:"string"}
BUCKET_NAME = os.environ.get("BUCKET_NAME", f"{PROJECT_ID}-cloudcorp-policies").strip()

# Guardrail: Prevent running with unconfigured placeholder Project ID
if PROJECT_ID in ["your-gcp-project-id", "YOUR_PROJECT_ID", ""]:
    raise ValueError(
        "⚠️ ACTION REQUIRED: Please enter your Google Cloud Project ID in the PROJECT_ID field above! "
        "Ensure Vertex AI and Cloud Storage APIs are enabled."
    )

# Prevent 403 ADC Quota Project error (e.g. consumer projects/618104708054 in Cloud Shell)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

# 3. Authentication Option A: Google Colab OAuth
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated via Google Colab OAuth.")
except ImportError:
    print("ℹ️ Running outside Google Colab (Cloud Shell / Local Machine / IDE).")

# 4. Authentication Option B: Service Account JSON Key via .env / Local File
sa_key_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "")
if sa_key_path:
    candidate_key_paths = [
        sa_key_path,
        os.path.join("..", sa_key_path),
        os.path.join("cloudcorp-llms-on-gcp-lab", sa_key_path),
        os.path.expanduser(f"~/Documents/{sa_key_path}"),
        os.path.expanduser(f"~/Downloads/{sa_key_path}")
    ]
    resolved_key = next((p for p in candidate_key_paths if os.path.isfile(p)), None)
    if resolved_key:
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.abspath(resolved_key)
        print(f"🔑 Loaded Service Account Key: {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")
    else:
        print(f"⚠️ Key file '{sa_key_path}' specified in .env not found on disk.")

# 5. Verify Active Credentials & Bind Quota Project
import google.auth
credentials = None
try:
    credentials, auth_project = google.auth.default(quota_project_id=PROJECT_ID)
    if hasattr(credentials, "with_quota_project"):
        credentials = credentials.with_quota_project(PROJECT_ID)
    account_id = getattr(credentials, "service_account_email", None) or getattr(credentials, "signer_email", "Google User Account (OAuth/ADC)")
    print(f"✅ Active Identity: {account_id}")
    print(f"✅ Quota Project bound: {PROJECT_ID}")
except Exception as e:
    print(f"⚠️ Warning: No active credentials detected ({e})")
    print("   • In Colab: Run auth.authenticate_user()")
    print("   • In Cloud Shell: Run 'gcloud auth application-default set-quota-project $PROJECT_ID'")
    print("   • Locally: Set GOOGLE_APPLICATION_CREDENTIALS in .env or run 'gcloud auth application-default login'")

# 6. Initialize Vertex AI & Cloud Storage with explicit credentials
import vertexai
vertexai.init(project=PROJECT_ID, location=REGION, credentials=credentials)
print(f"✅ Vertex AI initialized successfully for project '{PROJECT_ID}' in '{REGION}'.")

from google.cloud import storage
from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput
from vertexai.generative_models import GenerativeModel, FunctionDeclaration, Tool, Part

storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)
bucket = storage_client.bucket(BUCKET_NAME)
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

def chunk_document(source_name: str, text: str):
    sections = text.split("\n---")
    results = []
    for idx, sec in enumerate(sections):
        cleaned = sec.strip()
        if len(cleaned) > 20:
            results.append({"source": source_name, "chunk_id": f"{source_name}#sec{idx+1}", "text": cleaned})
    return results

print(f"Loading policy corpus from GCS bucket: gs://{BUCKET_NAME}...")
blobs = [b for b in bucket.list_blobs() if b.name.endswith(".txt")]
if not blobs:
    raise RuntimeError(
        f"❌ No policy files found in GCS bucket gs://{BUCKET_NAME}! "
        "Please run Section 1 first to ingest policy documents into GCS."
    )

all_chunks = []
for blob in blobs:
    chunks = chunk_document(blob.name, blob.download_as_text())
    all_chunks.extend(chunks)
    print(f" • Loaded '{blob.name}' from GCS -> {len(chunks)} sections")

print(f"Generating embeddings for {len(all_chunks)} chunks...")
doc_inputs = [TextEmbeddingInput(text=c["text"], task_type="RETRIEVAL_DOCUMENT") for c in all_chunks]
chunk_embeddings = embedding_model.get_embeddings(doc_inputs)
for i, emb in enumerate(chunk_embeddings):
    all_chunks[i]["vector"] = np.array(emb.values)

def retrieve_context(query: str, top_k: int = 1):
    query_input = TextEmbeddingInput(text=query, task_type="RETRIEVAL_QUERY")
    q_vec = np.array(embedding_model.get_embeddings([query_input])[0].values)
    scored = []
    for chunk in all_chunks:
        dot = np.dot(q_vec, chunk["vector"])
        norm = np.linalg.norm(q_vec) * np.linalg.norm(chunk["vector"])
        sim = float(dot / norm) if norm > 0 else 0.0
        scored.append((sim, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

print(f"✅ Loaded and indexed {len(all_chunks)} policy chunks from GCS for Agentic RAG reasoning.")


Loaded configuration from '../.env'
ℹ️ Running outside Google Colab (Local Machine / IDE).
🔑 Loaded Service Account Key: /Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/notebooks/gcp-key.json
✅ Active Identity: cloudcorp-lab-sa@upgradlabs-1751346914784.iam.gserviceaccount.com
✅ Vertex AI initialized successfully for project 'upgradlabs-1751346914784' in 'us-central1'.


/Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/.venv/lib/python3.12/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Loading policy corpus from GCS bucket: gs://upgradlabs-1751346914784-cloudcorp-policies...
 • Loaded 'hr_pto_policy.txt' from GCS -> 9 sections
 • Loaded 'it_laptop_policy.txt' from GCS -> 4 sections
 • Loaded 'remote_work_policy.txt' from GCS -> 4 sections
Generating embeddings for 17 chunks...
✅ Loaded and indexed 17 policy chunks from GCS for Agentic RAG reasoning.


In [8]:
# Step 3.2: Implement Mock Live IT Helpdesk Database & Tools
TICKETS_DB = {
    "INC-402": {"ticket_id": "INC-402", "status": "In Progress", "assigned_to": "Hardware Desk", "eta": "Tomorrow 2:00 PM"},
    "INC-105": {"ticket_id": "INC-105", "status": "Resolved", "assigned_to": "Network Ops", "eta": "Completed"}
}
TICKET_COUNTER = 501

def create_it_ticket(issue_summary: str, category: str = "Hardware Replacement") -> dict:
    """Tool: Creates a new IT incident ticket when an employee reports damaged/broken hardware."""
    global TICKET_COUNTER
    ticket_id = f"INC-{TICKET_COUNTER}"
    TICKET_COUNTER += 1
    record = {
        "ticket_id": ticket_id,
        "status": "Created (Approved for Loaner)",
        "category": category,
        "summary": issue_summary,
        "assigned_to": "Hardware Fulfillment Desk (Floor 2)",
        "priority": "High (P2)",
        "eta": "Within 24 hours",
        "instructions": "Present ticket ID at Floor 2 Tech Depot for emergency loaner laptop."
    }
    TICKETS_DB[ticket_id] = record
    return record

def lookup_it_ticket(ticket_id: str) -> dict:
    """Tool: Checks live status of an existing ticket."""
    tid = ticket_id.upper()
    return TICKETS_DB.get(tid, {"ticket_id": tid, "status": "Not Found", "message": "No matching ticket record."})

print("Tools implemented: create_it_ticket & lookup_it_ticket.")


Tools implemented: create_it_ticket & lookup_it_ticket.


In [12]:
# Step 3.3: Declare Tool Schemas for Gemini 3.5 Flash Lite
create_ticket_func = FunctionDeclaration(
    name="create_it_ticket",
    description="Creates a new IT incident ticket when an employee reports damaged, broken, or malfunctioning hardware.",
    parameters={
        "type": "object",
        "properties": {
            "issue_summary": {"type": "string", "description": "Concise summary of the reported hardware malfunction or damage."},
            "category": {"type": "string", "description": "Category of request, e.g., 'Hardware Replacement'."}
        },
        "required": ["issue_summary"]
    }
)

lookup_ticket_func = FunctionDeclaration(
    name="lookup_it_ticket",
    description="Fetches live progress and assignment status for an existing IT incident ticket using its ticket ID.",
    parameters={
        "type": "object",
        "properties": {
            "ticket_id": {"type": "string", "description": "The incident ticket ID (e.g., INC-402, INC-501)."}
        },
        "required": ["ticket_id"]
    }
)

agent_tools = Tool(function_declarations=[create_ticket_func, lookup_ticket_func])

# Initialize AskHR Agent with Tools & System Instructions on global endpoint
vertexai.init(project=PROJECT_ID, location="global", credentials=credentials)
agent_llm = GenerativeModel(
    "gemini-3.5-flash-lite",
    tools=[agent_tools],
    system_instruction=(
        "You are CloudCorp's AskHR & IT Agent.\n"
        "CRITICAL RULES:\n"
        "1. HARDWARE MALFUNCTION / DAMAGE: If an employee reports broken, damaged, or malfunctioning hardware (such as broken screen, defective battery, laptop damage), you MUST immediately call the  tool. Do NOT answer with plain text alone or just promise to create a ticket—invoke create_it_ticket immediately.\n"
        "2. TICKET STATUS LOOKUP: If an employee asks about an existing ticket status or ticket ID, you MUST call the  tool.\n"
        "3. GENERAL HR QUESTIONS: For general HR policy questions (such as remote work or PTO policy), answer directly from policy context without calling tools."
    )
)
print("✅ AskHR Agent initialized with external tools & deterministic tool-calling rules.")


✅ AskHR Agent initialized with external tools & deterministic tool-calling rules.


/Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [13]:
# Step 3.4: Autonomous Agent Execution Loop (3 Scenarios)
def run_askhr_agent(user_query: str):
    print()
    print(f"================ USER QUERY: '{user_query}' ================")
    vertexai.init(project=PROJECT_ID, location=REGION, credentials=credentials)
    # 1. Fetch relevant policy context via RAG
    rag_hits = retrieve_context(user_query, top_k=1)
    context_text = rag_hits[0][1]["text"] if rag_hits else "No specific policy chunk retrieved."
    
    vertexai.init(project=PROJECT_ID, location="global", credentials=credentials)
    # 2. Start Gemini conversation turn
    chat = agent_llm.start_chat()
    prompt = f"Policy Context:\n{context_text}\n\nUser Request: {user_query}"
    response = chat.send_message(prompt)
    
    part = response.candidates[0].content.parts[0]
    func_call = getattr(part, "function_call", None)
    
    # Check if Gemini decided to invoke a tool
    if func_call and getattr(func_call, "name", None):
        fn_name = func_call.name
        fn_args = dict(func_call.args)
        print(f"⚡ [AGENT TOOL CALL]: {fn_name}({fn_args})")
        
        # Execute tool function
        if fn_name == "create_it_ticket":
            tool_res = create_it_ticket(**fn_args)
        elif fn_name == "lookup_it_ticket":
            tool_res = lookup_it_ticket(**fn_args)
        else:
            tool_res = {"error": "Unknown tool"}
            
        print(f"📦 [TOOL RESULT]: {tool_res}")
        
        # Feed tool result back to Gemini using official Part.from_function_response
        response_part = Part.from_function_response(
            name=fn_name,
            response={"content": tool_res}
        )
        final_resp = chat.send_message(response_part)
        print(f"🤖 [FINAL SYNTHESIZED RESPONSE]:\n{final_resp.text.strip()}")
        return tool_res
    else:
        print("ℹ️ [PURE RAG PATH (No Tools Needed)]")
        print(f"🤖 [RESPONSE]:\n{response.text.strip()}")
        return None

# Test Scenario 1: Pure Policy (No Tool Needed)
run_askhr_agent("What is remote work policy?")

# Test Scenario 2: Equipment Malfunction -> RAG Guidance + Auto-Creates Ticket INC-501
created_ticket = run_askhr_agent("My laptop screen is broken and flickering, what should I do?")

# Test Scenario 3: Incident Status Check -> Dynamically checks the created ticket or pre-existing INC-402
ticket_to_check = created_ticket.get("ticket_id", "INC-402") if created_ticket else "INC-402"
run_askhr_agent(f"What is the status of my ticket {ticket_to_check}?")



================ USER QUERY: 'What is remote work policy?' ================
ℹ️ [PURE RAG PATH (No Tools Needed)]
🤖 [RESPONSE]:
Based on CloudCorp's Hybrid and Remote Work Policy (POL-REMOTE-2026-v2, revised January 20, 2026), the remote work policy applies to all global CloudCorp personnel and outlines the guidelines and expectations for hybrid and remote work arrangements. 

If you need specific details regarding eligibility, core working hours, equipment, or approval processes, please let me know!

================ USER QUERY: 'My laptop screen is broken and flickering, what should I do?' ================
⚡ [AGENT TOOL CALL]: create_it_ticket({'category': 'Hardware Replacement', 'issue_summary': 'Laptop screen is broken and flickering'})
📦 [TOOL RESULT]: {'ticket_id': 'INC-501', 'status': 'Created (Approved for Loaner)', 'category': 'Hardware Replacement', 'summary': 'Laptop screen is broken and flickering', 'assigned_to': 'Hardware Fulfillment Desk (Floor 2)', 'priority': 'High (P2

{'ticket_id': 'INC-501',
 'status': 'Created (Approved for Loaner)',
 'category': 'Hardware Replacement',
 'summary': 'Laptop screen is broken and flickering',
 'assigned_to': 'Hardware Fulfillment Desk (Floor 2)',
 'priority': 'High (P2)',
 'eta': 'Within 24 hours',
 'instructions': 'Present ticket ID at Floor 2 Tech Depot for emergency loaner laptop.'}